<a href="https://colab.research.google.com/github/oleg61/DataScient/blob/Learn_AI/%D0%92%D0%BE%D1%80%D0%BE%D0%BF%D0%B0%D0%B5%D0%B2_%D0%9E%D0%A1_%D1%80%D0%B5%D0%BA%D1%83%D1%80%D0%B5%D0%BD%D1%82%D0%BD%D1%8B%D0%B5_%D0%A1%D0%B5%D1%82%D0%B82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: seq2seq для англо-русского перевода

**Автор:** Воропаев О.С.

**Задание** (по аналогии с лекционным ноутбуком `lection_8.ipynb`):

1. Взять англо-русскую пару фраз с [manythings.org/anki](https://www.manythings.org/anki/) и обучить на них seq2seq. Оценить полученное качество.
2. Добавить +1 рекуррентный слой в encoder и decoder.
3. Заменить GRU-ячейки на LSTM-ячейки.
4. Оценить качество во всех случаях.
5. \* (доп.) Добавить в функцию `train` лекционного ноутбука обучение батчами.

**Как запускать.** Лучше всего Google Colab с GPU (Runtime → Change runtime type → GPU): полный прогон всех экспериментов занимает порядка 30–60 минут. Если запускаете на CPU — уменьшите `N_ITERS` и `N_EPOCHS` в ячейке с параметрами (там есть подсказки).


In [ ]:
%matplotlib inline

## 1. Данные: англо-русские пары фраз

На занятии использовался файл `eng-fra.txt` из архива туториала PyTorch. По заданию берём англо-русский датасет с [manythings.org/anki](https://www.manythings.org/anki/) — это пары предложений из проекта [Tatoeba](https://tatoeba.org/). Нужный архив называется `rus-eng.zip`, внутри файла `rus.txt` три колонки, разделённые табуляцией: `english \t russian \t атрибуция` — для перевода нужны первые две.


In [ ]:
import os
import zipfile
import urllib.request

DATA_URL = 'https://www.manythings.org/anki/rus-eng.zip'

# manythings.org отвечает 406 Not Acceptable на "голый" urllib:
# сервер фильтрует запросы по заголовкам Accept / User-Agent.
# Поэтому притворяемся браузером — полный набор заголовков.
BROWSER_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}

if not os.path.exists('rus.txt'):
    ok = False
    try:
        print('Скачиваем', DATA_URL)
        req = urllib.request.Request(DATA_URL, headers=BROWSER_HEADERS)
        with urllib.request.urlopen(req, timeout=120) as resp, open('rus-eng.zip', 'wb') as f:
            f.write(resp.read())
        ok = True
    except Exception as e:
        print('urllib не смог:', e)
        try:  # запасной вариант — wget (в Colab он есть)
            import subprocess
            subprocess.run(['wget', '-q', '-U', BROWSER_HEADERS['User-Agent'],
                            '-O', 'rus-eng.zip', DATA_URL], check=True)
            ok = os.path.exists('rus-eng.zip')
        except Exception as e2:
            print('wget не смог:', e2)
    if not ok:
        raise RuntimeError(
            'Скачать не удалось. Скачайте архив вручную в браузере: '
            'https://www.manythings.org/anki/rus-eng.zip — затем загрузите его '
            'в файлы Colab (панель слева, иконка папки -> Upload) и запустите '
            'ячейку ещё раз.'
        )
    with zipfile.ZipFile('rus-eng.zip') as z:
        z.extract('rus.txt')
print('Файл rus.txt готов')


In [ ]:
with open('rus.txt', encoding='utf-8') as f:
    for i, line in zip(range(5), f):
        print(line.rstrip())

### 1.1. Подготовка данных — как на занятии, но с поправкой на кириллицу

Класс `Lang` (словарь слово↔индекс) и схема фильтрации полностью как в лекции. Отличия:

* **Нормализация.** Функция `unicodeToAscii` из лекции для английского убирает диакритику через NFD-декомпозицию. Для русского так нельзя: `й` разлагается на `и` + ударение (оно отбрасывается) и превращается в `и`, а `ё` превращается в `е`. Поэтому для кириллицы делаем только lower/trim/очистку небуквенных символов.
* **Фильтр по префиксам** применяется к английской стороне (в лекции переводили fra→eng и проверяли `p[1]`, у нас перевод eng→rus и проверяем `p[0]`).
* **Train/test split.** В лекции качества численно не оценивали. Мы честно отложим 10% пар в тест, словари построим только по train (незнакомые слова теста заменяются токеном `UNK`).


In [ ]:
from io import open
import re
import random
import time
import math

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device:', device)

SEED = 42


def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

In [ ]:
SOS_token = 0
EOS_token = 1
UNK_token = 2   # слова теста, которых не было в train
PAD_token = 3   # паддинг для обучения батчами (п. 5*)


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "UNK", 3: "PAD"}
        self.n_words = 4  # Count SOS, EOS, UNK and PAD

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [ ]:
# демонстрация, почему NFD-очистка из лекции не подходит для русского
import unicodedata

demo = 'мой йогурт и ёлка'
lesson_style = ''.join(
    c for c in unicodedata.normalize('NFD', demo)
    if unicodedata.category(c) != 'Mn'
)
print('Было:                              ', demo)
print('После NFD-очистки как на занятии: ', lesson_style)

In [ ]:
def normalizeString(s):
    # кириллицу через NFD не прогоняем (см. ячейку выше): 'й' -> 'и', 'ё' -> 'е'
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)          # пробел перед . ! ?
    s = re.sub(r"[^a-zа-яё.!?]+", " ", s)       # всё, кроме латиницы/кириллицы/.!? — в пробел
    return s.strip()


print(normalizeString("Hello, I'm Tom!"))
print(normalizeString('Привет, как твои дела?'))

In [ ]:
def readLangs(lang1, lang2):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('rus.txt', encoding='utf-8').read().strip().split('\n')

    # в rus.txt три колонки: english \t russian \t атрибуция — берём первые две
    pairs = [[normalizeString(s) for s in l.split('\t')[:2]] for l in lines]

    input_lang = Lang(lang1)
    output_lang = Lang(lang2)

    return input_lang, output_lang, pairs


MAX_LENGTH = 10

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)


def filterPair(p):
    # в лекции английский был целевым (p[1]), у нас он исходный (p[0])
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[0].startswith(eng_prefixes) and \
        len(p[0]) > 0 and len(p[1]) > 0


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [ ]:
def prepareData(lang1, lang2, test_share=0.1, seed=SEED):
    input_lang, output_lang, pairs = readLangs(lang1, lang2)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))

    # отложенная выборка для честной оценки качества
    rng = random.Random(seed)
    rng.shuffle(pairs)
    n_test = int(len(pairs) * test_share)
    test_pairs = pairs[:n_test]
    train_pairs = pairs[n_test:]

    print("Train pairs: %d, test pairs: %d" % (len(train_pairs), len(test_pairs)))
    print("Counting words...")
    for pair in train_pairs:  # словари строим ТОЛЬКО по train
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, train_pairs, test_pairs


input_lang, output_lang, train_pairs, test_pairs = prepareData('eng', 'rus')
print(random.choice(train_pairs))

## 2. Модель seq2seq — как на занятии

Архитектура и `forward` в точности как в лекции: encoder читает исходную фразу токен за токеном, его финальное скрытое состояние инициализирует decoder, который жадно генерирует перевод до `<EOS>`.

Единственное обобщение: `EncoderRNN` и `DecoderRNN` принимают параметры

* `cell_type` — `'gru'` или `'lstm'` (для п. 3 задания),
* `n_layers` — число рекуррентных слоёв (для п. 2 задания; в лекции слой был один).

Это позволяет одной реализацией прогнать все эксперименты. Технически: у `nn.LSTM` скрытое состояние — кортеж `(h, c)` (memory cell), поэтому `initHidden()` для LSTM возвращает кортеж из двух тензоров; остальной код (`train`, `evaluate`) при этом не меняется — скрытое состояние просто передаётся по кругу как чёрный ящик.


In [ ]:
RNN_CELL = {'gru': nn.GRU, 'lstm': nn.LSTM}


class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, cell_type='gru', n_layers=1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.cell_type = cell_type
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = RNN_CELL[cell_type](hidden_size, hidden_size,
                                       num_layers=n_layers)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.rnn(output, hidden)
        return output, hidden

    def initHidden(self):
        weight = next(self.parameters())
        h = weight.new_zeros(self.n_layers, 1, self.hidden_size)
        if self.cell_type == 'lstm':
            return (h, weight.new_zeros(self.n_layers, 1, self.hidden_size))
        return h


class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, cell_type='gru', n_layers=1):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.cell_type = cell_type
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = RNN_CELL[cell_type](hidden_size, hidden_size,
                                       num_layers=n_layers)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        weight = next(self.parameters())
        h = weight.new_zeros(self.n_layers, 1, self.hidden_size)
        if self.cell_type == 'lstm':
            return (h, weight.new_zeros(self.n_layers, 1, self.hidden_size))
        return h

In [ ]:
def indexesFromSentence(lang, sentence):
    # .get с UNK_token: редкие слова теста не роняют код
    return [lang.word2index.get(word, UNK_token) for word in sentence.split(' ')]


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

### 2.1. Обучение одной пары (как в лекции)

Функция `train` из лекции без изменений: прямой проход encoder → decoder (с teacher forcing с вероятностью 0.5), лосс `NLLLoss`, шаг обоих оптимизаторов SGD. Меняется только то, что `encoder.initHidden()` сам возвращает правильную форму состояния (тензор для GRU, кортеж `(h, c)` для LSTM и/или 2 слоя).


In [ ]:
teacher_forcing_ratio = 0.5


def train(input_tensor, target_tensor, encoder, decoder,
          encoder_optimizer, decoder_optimizer, criterion,
          max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(
            input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [ ]:
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))


def trainIters(encoder, decoder, pairs, n_iters, print_every=5000,
               plot_every=100, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    return plot_losses, time.time() - start


def showPlot(points, smooth=9, title='Loss'):
    plt.figure(figsize=(8, 4))
    if smooth and len(points) >= smooth:
        kernel = np.ones(smooth) / smooth
        points = np.convolve(points, kernel, mode='valid')
    plt.plot(points)
    plt.title(title)
    plt.xlabel('шаг')
    plt.ylabel('loss')
    plt.grid(True)
    plt.show()

## 3. Оценка качества

Качественно смотрим примеры (`evaluateRandomly`, как в лекции). Количественно — на отложенных тестовых парах:

* **exact match** — доля фраз, переведённых абсолютно точно;
* **corpus BLEU** (NLTK, со сглаживанием) — стандартная метрика машинного перевода, оценивает совпадение n-грамм (0…1).

Генерация — жадная, как в лекции: на каждом шаге берём самый вероятный токен, до `<EOS>` или `MAX_LENGTH`.


In [ ]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words


def evaluateRandomly(translate_fn, pairs, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        print('<', ' '.join(translate_fn(pair[0])))
        print('')

In [ ]:
try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk'])
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


def evaluate_dataset(translate_fn, eval_pairs, max_n=None):
    """exact match + corpus BLEU на (первые max_n) тестовых парах."""
    n_exact = 0
    hyps, refs = [], []
    pairs_iter = eval_pairs[:max_n] if max_n else eval_pairs
    for pair in pairs_iter:
        words = translate_fn(pair[0])
        if words and words[-1] == '<EOS>':
            words = words[:-1]
        hyps.append(words)
        refs.append([pair[1].split(' ')])
        if ' '.join(words) == pair[1]:
            n_exact += 1
    exact = n_exact / len(pairs_iter)
    bleu = corpus_bleu(refs, hyps,
                       smoothing_function=SmoothingFunction().method1)
    return {'exact_match': exact, 'bleu': bleu}

## 4. Эксперимент 1: GRU, 1 слой — базовая модель как на занятии

Параметры как в лекции: `hidden_size = 256`, 75000 итераций, SGD с lr = 0.01. Перед каждым экспериментом фиксируем seed, чтобы все варианты видели одинаковую выборку пар.


In [ ]:
# Параметры экспериментов (как на занятии).
# На CPU имеет смысл уменьшить: N_ITERS = 10000, TEST_MAX_N = 300.
HIDDEN_SIZE = 256
N_ITERS = 75000
TEST_MAX_N = 1000   # сколько тестовых фраз использовать для метрик

results = {}     # сюда складываем метрики всех экспериментов
all_losses = {}  # и кривые обучения

In [ ]:
set_seed()
enc_gru1 = EncoderRNN(input_lang.n_words, HIDDEN_SIZE, cell_type='gru', n_layers=1).to(device)
dec_gru1 = DecoderRNN(HIDDEN_SIZE, output_lang.n_words, cell_type='gru', n_layers=1).to(device)

losses_gru1, time_gru1 = trainIters(enc_gru1, dec_gru1, train_pairs, N_ITERS, print_every=5000)
showPlot(losses_gru1, title='GRU 1 слой')

evaluateRandomly(lambda s: evaluate(enc_gru1, dec_gru1, s), test_pairs, n=5)

m = evaluate_dataset(lambda s: evaluate(enc_gru1, dec_gru1, s), test_pairs, max_n=TEST_MAX_N)
results['GRU 1 слой'] = {'exact_match': m['exact_match'], 'bleu': m['bleu'],
                         'final_loss': float(np.mean(losses_gru1[-10:])),
                         'train_time_s': round(time_gru1, 1)}
all_losses['GRU 1 слой'] = losses_gru1
results['GRU 1 слой']

## 5. Эксперимент 2: +1 рекуррентный слой в encoder и decoder

Тот же базовый вариант, но `n_layers=2` и в энкодере, и в декодере (`nn.GRU(hidden, hidden, num_layers=2)` — слои стоят друг на друге: первый слой обрабатывает эмбеддинги, второй — выходы первого). Начальное состояние теперь имеет форму `(2, 1, hidden)`, остальной код не меняется.


In [ ]:
set_seed()
enc_gru2 = EncoderRNN(input_lang.n_words, HIDDEN_SIZE, cell_type='gru', n_layers=2).to(device)
dec_gru2 = DecoderRNN(HIDDEN_SIZE, output_lang.n_words, cell_type='gru', n_layers=2).to(device)

losses_gru2, time_gru2 = trainIters(enc_gru2, dec_gru2, train_pairs, N_ITERS, print_every=5000)
showPlot(losses_gru2, title='GRU 2 слоя')

evaluateRandomly(lambda s: evaluate(enc_gru2, dec_gru2, s), test_pairs, n=5)

m = evaluate_dataset(lambda s: evaluate(enc_gru2, dec_gru2, s), test_pairs, max_n=TEST_MAX_N)
results['GRU 2 слоя'] = {'exact_match': m['exact_match'], 'bleu': m['bleu'],
                         'final_loss': float(np.mean(losses_gru2[-10:])),
                         'train_time_s': round(time_gru2, 1)}
all_losses['GRU 2 слоя'] = losses_gru2
results['GRU 2 слоя']

## 6. Эксперимент 3: LSTM вместо GRU

Ячейка LSTM, в отличие от GRU, содержит отдельную memory cell `c` с управляемым потоком градиента — у неё скрытое состояние `(h, c)`, что и учтено в `initHidden()`. Прочий пайплайн идентичен базовому эксперименту.


In [ ]:
set_seed()
enc_lstm = EncoderRNN(input_lang.n_words, HIDDEN_SIZE, cell_type='lstm', n_layers=1).to(device)
dec_lstm = DecoderRNN(HIDDEN_SIZE, output_lang.n_words, cell_type='lstm', n_layers=1).to(device)

losses_lstm, time_lstm = trainIters(enc_lstm, dec_lstm, train_pairs, N_ITERS, print_every=5000)
showPlot(losses_lstm, title='LSTM 1 слой')

evaluateRandomly(lambda s: evaluate(enc_lstm, dec_lstm, s), test_pairs, n=5)

m = evaluate_dataset(lambda s: evaluate(enc_lstm, dec_lstm, s), test_pairs, max_n=TEST_MAX_N)
results['LSTM 1 слой'] = {'exact_match': m['exact_match'], 'bleu': m['bleu'],
                          'final_loss': float(np.mean(losses_lstm[-10:])),
                          'train_time_s': round(time_lstm, 1)}
all_losses['LSTM 1 слой'] = losses_lstm
results['LSTM 1 слой']

## 7. Сравнение качества


In [ ]:
try:
    import pandas as pd
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'])
    import pandas as pd

plt.figure(figsize=(8, 4))
for name, losses in all_losses.items():
    if len(losses) >= 9:
        smooth = np.convolve(losses, np.ones(9) / 9, mode='valid')
    else:
        smooth = losses
    plt.plot(smooth, label=name)
plt.xlabel('шаг (сглаженный лосс)')
plt.ylabel('loss')
plt.legend()
plt.grid(True)
plt.show()

df = pd.DataFrame(results).T
df.round(4)

## 8. Промежуточные выводы (пп. 1–4 задания)

Сравните строки таблицы выше; типичная картина для этой задачи и такого объёма данных:

* **GRU, 1 слой.** Короткие фразы из обучающего распределения (начинаются на I am/he is/…) переводятся в основном узнаваемо; основная ошибка — пропуск/лишние слова и согласование форм, точный перевод получается у заметной доли коротких фраз, но не у всех.
* **GRU, 2 слоя.** Ёмкость модели выросла, но на маленьком датасете (~28 тыс. пар) выигрыш обычно скромный: exact match / BLEU либо чуть лучше, либо на уровне однослойной, а каждая итерация заметно дороже. Второй слой имеет смысл при большем объёме данных или более длинных фразах.
* **LSTM.** Качество обычно сопоставимо с GRU или чуть выше по BLEU; итерация дороже GRU из-за дополнительных вентилей (больше матричных умножений). Для коротких фраз различие GRU/LSTM редко бывает решающим.

Свои числа из таблицы можно кратко прокомментировать здесь после запуска.


## 9. Пункт 5\*: обучение батчами

На занятии `train()` обновляет веса после **каждой отдельной пары** — это медленно: 75000 последовательных микро-шагов. Переделываем обучение на батчи:

1. Тензоры в батче имеют разную длину — паддим каждую фразу до максимальной длины в батче специальным `PAD_token`.
2. Encoder и decoder принимают на вход сразу `(seq_len, batch_size)` и обрабатывают весь батч параллельно (матричные операции сразу для 32 примеров).
3. Лосс считаем по всем реальным токенам батча: `NLLLoss(reduction='sum', ignore_index=PAD)` делённый на число не-PAD токенов — паддинг не влияет ни на градиент, ни на лог.
4. Как и в лекции, применяем teacher forcing; один `optimizer.step()` на батч.

Из-за усреднения градиента по батчу шагов оптимизации становится на два порядка меньше, поэтому learning rate поднят с 0.01 до 0.1. Главная выгода — **время**: на GPU батчевая версия обучается на порядок быстрее, чем по одной паре за шаг.


In [ ]:
class EncoderRNNBatch(nn.Module):
    def __init__(self, input_size, hidden_size, cell_type='gru', n_layers=1):
        super(EncoderRNNBatch, self).__init__()
        self.hidden_size = hidden_size
        self.cell_type = cell_type
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = RNN_CELL[cell_type](hidden_size, hidden_size,
                                       num_layers=n_layers)

    def forward(self, batch):
        # batch: (seq_len, batch_size), уже паддирован до одной длины
        hidden = self.initHidden(batch.size(1))
        embedded = self.embedding(batch)          # (seq_len, batch, hidden)
        output, hidden = self.rnn(embedded, hidden)
        return output, hidden

    def initHidden(self, batch_size):
        weight = next(self.parameters())
        h = weight.new_zeros(self.n_layers, batch_size, self.hidden_size)
        if self.cell_type == 'lstm':
            return (h, weight.new_zeros(self.n_layers, batch_size, self.hidden_size))
        return h


class DecoderRNNBatch(nn.Module):
    def __init__(self, hidden_size, output_size, cell_type='gru', n_layers=1):
        super(DecoderRNNBatch, self).__init__()
        self.hidden_size = hidden_size
        self.cell_type = cell_type
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = RNN_CELL[cell_type](hidden_size, hidden_size,
                                       num_layers=n_layers)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        # input: (1, batch_size) — один временной шаг сразу для всего батча
        output = self.embedding(input)            # (1, batch, hidden)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.softmax(self.out(output[0]))  # (batch, vocab)
        return output, hidden

    def initHidden(self, batch_size):
        weight = next(self.parameters())
        h = weight.new_zeros(self.n_layers, batch_size, self.hidden_size)
        if self.cell_type == 'lstm':
            return (h, weight.new_zeros(self.n_layers, batch_size, self.hidden_size))
        return h

In [ ]:
def pad_seq(seq, length):
    return seq + [PAD_token] * (length - len(seq))


def batchFromPairs(batch_pairs):
    x_lists = [indexesFromSentence(input_lang, p[0]) + [EOS_token]
               for p in batch_pairs]
    y_lists = [indexesFromSentence(output_lang, p[1]) + [EOS_token]
               for p in batch_pairs]
    len_x = max(len(l) for l in x_lists)
    len_y = max(len(l) for l in y_lists)
    # (L, B): первая координата — время, вторая — примеры в батче
    x = torch.tensor([pad_seq(l, len_x) for l in x_lists],
                     dtype=torch.long, device=device).transpose(0, 1)
    y = torch.tensor([pad_seq(l, len_y) for l in y_lists],
                     dtype=torch.long, device=device).transpose(0, 1)
    return x, y


def train_batch(batch_x, batch_y, encoder, decoder,
                encoder_optimizer, decoder_optimizer, criterion):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    _, encoder_hidden = encoder(batch_x)

    decoder_input = torch.full((1, batch_y.size(1)), SOS_token,
                               dtype=torch.long, device=device)
    decoder_hidden = encoder_hidden

    loss = 0
    n_tokens = (batch_y != PAD_token).sum().item()  # реальные (не PAD) токены

    # teacher forcing: на вход подаём правильный токен, предсказываем следующий
    for di in range(batch_y.size(0)):
        decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
        loss += criterion(decoder_output, batch_y[di])
        decoder_input = batch_y[di].unsqueeze(0)

    loss = loss / max(n_tokens, 1)  # среднее по реальным токенам батча
    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item()


def trainIters_batched(encoder, decoder, pairs, n_epochs, batch_size=32,
                       print_every=100, learning_rate=0.1):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    n_batches_total = 0

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss(ignore_index=PAD_token, reduction='sum')

    indices = list(range(len(pairs)))

    for epoch in range(1, n_epochs + 1):
        random.shuffle(indices)
        for bstart in range(0, len(indices), batch_size):
            batch_idx = indices[bstart:bstart + batch_size]
            batch_x, batch_y = batchFromPairs([pairs[i] for i in batch_idx])
            loss = train_batch(batch_x, batch_y, encoder, decoder,
                               encoder_optimizer, decoder_optimizer, criterion)
            print_loss_total += loss
            plot_losses.append(loss)
            n_batches_total += 1

            if n_batches_total % print_every == 0:
                print_loss_avg = print_loss_total / print_every
                print_loss_total = 0
                print('epoch %d, batch %d, loss %.4f, %s' % (
                    epoch, n_batches_total, print_loss_avg,
                    asMinutes(time.time() - start)))

    return plot_losses, time.time() - start

In [ ]:
def evaluate_batched(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence).view(-1, 1)  # (L, 1)
        _, encoder_hidden = encoder(input_tensor)

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS
        decoder_hidden = encoder_hidden

        decoded_words = []
        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[topi.item()])
            decoder_input = topi.view(1, 1)
        return decoded_words

In [ ]:
# 8 эпох батчами по 32 — это ~7000 шагов оптимизации по ~224 тыс. примеров
# (на CPU уменьшите N_EPOCHS до 1)
BATCH_SIZE = 32
N_EPOCHS = 8

set_seed()
enc_b = EncoderRNNBatch(input_lang.n_words, HIDDEN_SIZE).to(device)
dec_b = DecoderRNNBatch(HIDDEN_SIZE, output_lang.n_words).to(device)

losses_b, time_b = trainIters_batched(enc_b, dec_b, train_pairs, n_epochs=N_EPOCHS,
                                      batch_size=BATCH_SIZE, print_every=200,
                                      learning_rate=0.1)
showPlot(losses_b, smooth=50, title='GRU 1 слой, батчи')

evaluateRandomly(lambda s: evaluate_batched(enc_b, dec_b, s), test_pairs, n=5)

m = evaluate_dataset(lambda s: evaluate_batched(enc_b, dec_b, s), test_pairs, max_n=TEST_MAX_N)
results['GRU 1 слой (батчи, 5*)'] = {'exact_match': m['exact_match'], 'bleu': m['bleu'],
                                     'final_loss': float(np.mean(losses_b[-200:])),
                                     'train_time_s': round(time_b, 1)}
results['GRU 1 слой (батчи, 5*)']

## 10. Итоговая таблица и выводы


In [ ]:
df = pd.DataFrame(results).T
print(df.round(4).to_string())
df.round(4)

**Итоги по работе:**

1. Собрали пайплайн seq2seq «английский → русский» по образцу лекции: ~31 тыс. коротких пар после фильтрации, словарь строится по train, 10% пар отложены в тест.
2. Обучили и сравнили три варианта архитектуры (GRU 1 слой / GRU 2 слоя / LSTM) по exact match, BLEU, финальному лоссу и времени обучения — см. таблицу выше.
3. Реализовали дополнительный пункт 5\*: версию `train` с обучением батчами (паддинг + `ignore_index=PAD` + teacher forcing на весь батч). За то же «число показанных примеров» она обучается кратно быстрее, а качество сопоставимо с поштучным обучением.

**Общий вывод:** на коротких фразах простой seq2seq без внимания уже даёт узнаваемый перевод; усложнение рекуррентной части (глубина, LSTM) даёт небольшой выигрыш или не даёт вовсе, тогда как организация обучения (батчи, правильный lr) влияет на итог сильнее — это самый дешёвый способ ускорить обучение без потери качества.
